# 005 — Temperature & Degree Days (HDD / CDD)

**Source:** Open-Meteo historical archive API (ERA5 reanalysis)  
**File:** `data/raw/temperatures_daily.csv`  
**Units:** °C (5-city national average); derived HDD/CDD in degree-days base 65°F  
**Frequency:** Daily  
**Coverage:** 2000-01-01 to present  
**Cities averaged:** New York, Chicago, Houston, Atlanta, Boston

**Goal:** Understand the temperature signal that drives US gas demand — its seasonal structure, year-to-year variability, and the HDD/CDD transformation that converts raw temperature into the demand-relevant units the market uses. Temperature is the primary short-term driver of gas consumption.

In [ ]:
from pathlib import Path

project_root = Path("/Users/alexbrown/Desktop/Work EX /GIC/NatGasModel_US")
print("Project root:", project_root)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns

sns.set_theme(context="paper", font_scale=2.0, style="whitegrid")

DIAG = project_root / "diagram" / "005_temperature"
DIAG.mkdir(parents=True, exist_ok=True)

## 1. Load & Inspect

In [ ]:
df = pd.read_csv(project_root / "data" / "raw" / "temperatures_daily.csv", index_col="date", parse_dates=True)
temp_c = df["temperature_c"].astype(float)

# Drop trailing rows with no data (Open-Meteo lag on most recent days)
temp_c = temp_c.dropna()

print(f"Date range:   {temp_c.index.min().date()} → {temp_c.index.max().date()}")
print(f"Observations: {len(temp_c):,}")
print(f"Missing:      {temp_c.isna().sum()}")

In [ ]:
temp_c.to_frame().sample(5).sort_index().round(2)

In [ ]:
temp_c.describe().round(2)

## 2. Convert to Fahrenheit & Derive HDD / CDD

The US market uses **base 65°F** for both HDD and CDD — the temperature at which neither heating nor cooling is needed.

- **HDD (Heating Degree Day):** `max(0, 65 - T_F)` — measures how cold a day is relative to the comfort threshold. Directly proportional to heating fuel demand
- **CDD (Cooling Degree Day):** `max(0, T_F - 65)` — measures how hot a day is. Drives air conditioning load, which means gas-fired power burn

In [ ]:
temp_f = temp_c * 9 / 5 + 32
temp_f.name = "temperature_f"

HDD = (65 - temp_f).clip(lower=0)
HDD.name = "HDD"

CDD = (temp_f - 65).clip(lower=0)
CDD.name = "CDD"

weather = pd.DataFrame({"temp_c": temp_c, "temp_f": temp_f, "HDD": HDD, "CDD": CDD})

print(weather.describe().round(2))

## 3. Full Temperature History

In [ ]:
fig, ax = plt.subplots(figsize=(14, 4))

temp_f.plot(ax=ax, color="tab:red", alpha=0.3, linewidth=0.6, label="Daily (°F)")
temp_f.rolling(30).mean().plot(ax=ax, color="tab:red", linewidth=1.5, label="30-day rolling mean")
ax.axhline(65, color="black", linewidth=0.8, linestyle="--", alpha=0.5, label="65°F comfort threshold")

events = {
    "2014-01-07": "Polar vortex",
    "2019-01-30": "Polar vortex II",
    "2021-02-15": "Winter Storm Uri",
    "2023-12-20": "Dec 2023 cold snap",
}

for date, label in events.items():
    ax.axvline(pd.Timestamp(date), color="steelblue", alpha=0.5, linewidth=0.8)
    ax.annotate(
        label,
        xy=(pd.Timestamp(date), temp_f.max() * 0.95),
        fontsize=8,
        rotation=90,
        va="top",
        ha="right",
        color="steelblue",
        alpha=0.8,
    )

ax.set_title("5-City Average Temperature (NY, Chicago, Houston, Atlanta, Boston) — Full History")
ax.set_ylabel("°F")
ax.set_xlabel("")
ax.legend(loc="upper left")

fig.tight_layout()
fig.savefig(DIAG / "temperature_full_history.svg", format="svg", bbox_inches="tight")
plt.show()

## 4. Recent Period — 2020 to Present

In [ ]:
recent_f = temp_f["2020":]

fig, ax = plt.subplots(figsize=(14, 4))

recent_f.plot(ax=ax, color="tab:red", alpha=0.3, linewidth=0.6, label="Daily (°F)")
recent_f.rolling(30).mean().plot(ax=ax, color="tab:red", linewidth=1.5, label="30-day rolling mean")
ax.axhline(65, color="black", linewidth=0.8, linestyle="--", alpha=0.5, label="65°F threshold")

recent_events = {
    "2021-02-15": "Uri",
    "2022-12-20": "Dec 2022 bomb cyclone",
    "2023-12-20": "Dec 2023 cold snap",
}

for date, label in recent_events.items():
    ax.axvline(pd.Timestamp(date), color="steelblue", alpha=0.5, linewidth=0.8)
    ax.annotate(
        label,
        xy=(pd.Timestamp(date), recent_f.max() * 0.95),
        fontsize=9,
        rotation=90,
        va="top",
        ha="right",
        color="steelblue",
        alpha=0.8,
    )

ax.set_title("5-City Average Temperature — 2020 to Present")
ax.set_ylabel("°F")
ax.set_xlabel("")
ax.legend(loc="upper left")

fig.tight_layout()
fig.savefig(DIAG / "temperature_recent_2020.svg", format="svg", bbox_inches="tight")
plt.show()

## 5. HDD & CDD — Full History

HDD and CDD are how the market communicates temperature deviations in demand-relevant units. When a forecaster says "this week was 50 HDD above normal", that maps directly to an expected storage draw surprise.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True)

# Monthly HDD sum — more readable than daily bars over 25 years
hdd_monthly = HDD.resample("ME").sum()
cdd_monthly = CDD.resample("ME").sum()

axes[0].bar(hdd_monthly.index, hdd_monthly.values, color="steelblue", alpha=0.7, width=20, label="Monthly HDD")
axes[0].set_ylabel("HDD (monthly sum)")
axes[0].set_title("Monthly Heating Degree Days (HDD) and Cooling Degree Days (CDD)")
axes[0].legend(loc="upper right")

axes[1].bar(cdd_monthly.index, cdd_monthly.values, color="tab:orange", alpha=0.7, width=20, label="Monthly CDD")
axes[1].set_ylabel("CDD (monthly sum)")
axes[1].set_xlabel("")
axes[1].legend(loc="upper right")

fig.tight_layout()
fig.savefig(DIAG / "hdd_cdd_full_history.svg", format="svg", bbox_inches="tight")
plt.show()

## 6. Seasonal Profile — Monthly Average HDD & CDD

In [ ]:
month_labels = ["Jan", "Feb", "Mar", "Apr", "May", "Jun",
                "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]

# Average monthly sum by calendar month
hdd_monthly_df = hdd_monthly.to_frame(name="HDD")
hdd_monthly_df["month"] = hdd_monthly_df.index.month
cdd_monthly_df = cdd_monthly.to_frame(name="CDD")
cdd_monthly_df["month"] = cdd_monthly_df.index.month

avg_hdd = hdd_monthly_df.groupby("month")["HDD"].mean()
avg_cdd = cdd_monthly_df.groupby("month")["CDD"].mean()

fig, ax = plt.subplots(figsize=(12, 4))

x = np.arange(1, 13)
width = 0.4

ax.bar(x - width / 2, avg_hdd.values, width=width, color="steelblue", alpha=0.8, label="Avg HDD")
ax.bar(x + width / 2, avg_cdd.values, width=width, color="tab:orange", alpha=0.8, label="Avg CDD")

ax.set_xticks(x)
ax.set_xticklabels(month_labels)
ax.set_title("Average Monthly HDD and CDD by Month (2000–present, base 65°F)")
ax.set_ylabel("Degree days")
ax.legend()

fig.tight_layout()
fig.savefig(DIAG / "hdd_cdd_seasonal_profile.svg", format="svg", bbox_inches="tight")
plt.show()

## 7. Year-to-Year HDD Variability

Annual HDD total shows which winters were cold (high demand) vs warm (low demand). The spread year-to-year is what creates storage surprises and price spikes.

In [ ]:
# Annual totals — use gas year (Oct–Sep) to keep a winter together
# Simple approach: calendar year HDD/CDD totals
annual_hdd = HDD.resample("YE").sum()
annual_cdd = CDD.resample("YE").sum()

# Drop the current partial year
current_year = pd.Timestamp.today().year
annual_hdd = annual_hdd[annual_hdd.index.year < current_year]
annual_cdd = annual_cdd[annual_cdd.index.year < current_year]

fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True)

axes[0].bar(annual_hdd.index.year, annual_hdd.values, color="steelblue", alpha=0.8)
axes[0].axhline(annual_hdd.mean(), color="black", linewidth=1.0, linestyle="--", label=f"Mean: {annual_hdd.mean():.0f}")
axes[0].set_ylabel("Annual HDD")
axes[0].set_title("Annual HDD and CDD Totals — Year-to-Year Variability")
axes[0].legend()

axes[1].bar(annual_cdd.index.year, annual_cdd.values, color="tab:orange", alpha=0.8)
axes[1].axhline(annual_cdd.mean(), color="black", linewidth=1.0, linestyle="--", label=f"Mean: {annual_cdd.mean():.0f}")
axes[1].set_ylabel("Annual CDD")
axes[1].set_xlabel("Year")
axes[1].legend()

fig.tight_layout()
fig.savefig(DIAG / "hdd_cdd_annual_variability.svg", format="svg", bbox_inches="tight")
plt.show()

In [ ]:
# Print the extreme years for context
print("Coldest winters (highest HDD):")
print(annual_hdd.sort_values(ascending=False).head(5).rename(lambda x: x.year).to_string())
print()
print("Warmest winters (lowest HDD):")
print(annual_hdd.sort_values().head(5).rename(lambda x: x.year).to_string())
print()
print("Hottest summers (highest CDD):")
print(annual_cdd.sort_values(ascending=False).head(5).rename(lambda x: x.year).to_string())

## 8. Observations

### Why temperature is the primary demand driver

Gas demand in the US is dominantly weather-driven on short time horizons:

- **Residential and commercial heating** is switched on and off by temperature. Below 65°F, each additional degree colder adds a roughly linear amount of gas demand. The relationship is strong and immediate — a cold day adds demand that same day
- **Power burn for cooling** (air conditioning) works the same way on the hot side. Above 65°F, cooling load increases, driving gas-fired generation (the marginal source in most regions when demand peaks)
- **Industrial demand** is relatively insensitive to short-term temperature — it's driven by economic activity and contracts, not weather

The result: if you know temperature, you can explain ~60–70% of week-to-week gas demand variation in winter.

---

### The HDD/CDD transformation — why not just use raw temperature

Raw temperature (°F) is a symmetric signal — a day at 40°F and a day at 90°F are equally far from average, but their demand implications are different in magnitude and sector:

- **HDD** focuses entirely on the heating side. A day at 50°F = 15 HDD. A day at 30°F = 35 HDD. Below 65°F is when gas consumption responds; above 65°F heating demand is essentially zero
- **CDD** focuses on the cooling side. A day at 85°F = 20 CDD. A day at 65°F = 0 CDD

This non-linearity (the `clip(lower=0)`) is what makes HDD/CDD more useful as model features than raw temperature — they encode the threshold behaviour of heating and cooling equipment.

---

### The US two-season structure in the data

The seasonal profile chart shows the key asymmetry of the US market:

- **HDD dominates Dec–Feb** — the winter heating peak. January has the highest average HDD; demand is primarily residential and commercial heating
- **CDD peaks Jun–Aug** — the summer cooling peak. This is when power burn for air conditioning drives gas demand in the South and Midwest
- **Spring (Mar–May) and autumn (Sep–Nov) are shoulder months** — both HDD and CDD near zero. This is the injection season for storage; demand is at its seasonal floor

This differs structurally from European markets (Germany, UK) which have an almost single-season demand profile — heating dominates and there is no comparable summer cooling load because European buildings rarely have air conditioning at scale.

---

### Year-to-year variability — the source of storage surprises

The annual HDD chart shows substantial variability. The coldest vs warmest winters in this dataset differ by ~500–800 HDD — a difference that translates to hundreds of Bcf in cumulative storage draws over a winter season.

This variability is the fundamental source of uncertainty in gas price forecasting:

| Winter type | HDD vs normal | Storage impact | Price impact |
|---|---|---|---|
| Colder than normal | +HDD | Larger draws → lower end-of-winter inventory | Bullish |
| Warmer than normal | -HDD | Smaller draws → excess inventory entering spring | Bearish |
| Normal | ~avg HDD | Storage tracks 5-year average | Neutral |

The polar vortex events (2014, 2019) and Winter Storm Uri (2021) are visible as extreme cold outliers. A 10-day extreme cold event like Uri generates more HDD than an entire typical April — and it hits all at once, overwhelming the storage drawdown rate.

---

### The 5-city average as a national consumption proxy

The temperature series here is a simple equal-weight average of New York, Chicago, Houston, Atlanta, and Boston. This is a reasonable approximation for a national demand signal, but it has limitations:

- **Northeast and Midwest** (NY, Chicago, Boston) are heating-dominated. Cold winters there drive the largest demand spikes because population density is high and heating gas use is intense
- **Houston (South)** is cooling-dominated. Hot summers there drive power burn, not heating
- **Population weighting** would be more accurate — but equal-weight gets the seasonal structure right

For the model feature matrix: HDD and CDD derived from this series are the primary weather regressors. Lagged HDD (1–7 days) can capture the slow response of building thermal mass. Monthly accumulated HDD/CDD maps naturally onto monthly consumption data.